# Gödel trace v3 — pretrained model, no training

הגרסה הזו מחליפה את Pythia ב־`distilgpt2` כדי להימנע מה־NaN שקיבלנו בקריאת attention.

אין כאן אימון. המודל רק מריץ forward pass.

בנוסף, כדי שהקוד יתאר *חישוב מסוים* ולא רק את הארכיטקטורה, בכל head נשמור באופן דטרמיניסטי את שני מקורות ה־attention החזקים ביותר. המשקל עצמו לא נכנס לקוד.

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"
PROMPT = "The dog chased the red ball."

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager",
    torch_dtype=torch.float32,
).to(device)
model.eval()

inputs = tokenizer(PROMPT, return_tensors="pt").to(device)

with torch.no_grad():
    out = model(
        **inputs,
        output_attentions=True,
        output_hidden_states=True,
        use_cache=False,
        return_dict=True,
    )

tokens = inputs["input_ids"][0].tolist()
str_tokens = tokenizer.convert_ids_to_tokens(tokens)
next_id = int(out.logits[0,-1].argmax())

print("device:", device)
print("tokens:", list(enumerate(str_tokens)))
print("prediction:", repr(tokenizer.decode([next_id])))
print("layers:", len(out.attentions), "heads:", out.attentions[0].shape[1])

bad = []
for l, a in enumerate(out.attentions):
    if not torch.isfinite(a).all():
        bad.append(l)

print("layers with non-finite attention:", bad)
assert not bad, f"Attention extraction failed in layers {bad}"
print("✓ כל מטריצות ה-attention תקינות")

## בונים trace רקורסיבי

קודי הפעולות נקבעים מראש. אין classifier ואין למידה נוספת.

אנחנו לא מחשבים את מספר גדל הענק; כל צומת שומר `(operation, parents)` — כלומר את הפירוק הפורמלי שלו.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Any

OP = {
    "TOKEN": 1,
    "EMBED": 2,
    "ATTEND": 3,
    "HEAD_MERGE": 4,
    "MLP": 5,
    "RESIDUAL_ADD": 6,
    "FINAL": 7,
}

@dataclass
class GNode:
    idx: int
    op: str
    parents: List[int] = field(default_factory=list)
    meta: Dict[str, Any] = field(default_factory=dict)

nodes=[]

def add(op, parents=None, **meta):
    n=GNode(len(nodes), op, list(parents or []), meta)
    nodes.append(n)
    return n.idx

# עלי העץ: הטוקנים וה-embeddings
resid=[]
for pos, tok in enumerate(tokens):
    t=add("TOKEN", [], token_id=tok, text=str_tokens[pos], position=pos)
    e=add("EMBED", [t], position=pos)
    resid.append(e)

TOP_K = 2
L=len(out.attentions)
H=out.attentions[0].shape[1]
T=len(tokens)

for layer in range(L):
    next_resid=[]
    for target in range(T):
        head_nodes=[]
        for head in range(H):
            row=out.attentions[layer][0, head, target, :target+1]
            k=min(TOP_K, row.numel())
            vals, inds=torch.topk(row, k)

            # שני המקורות החזקים ביותר בלבד.
            # הערכים עצמם נשמרים כ-metadata ולא כחלק מקוד גדל.
            parents=[resid[int(s)] for s in inds.tolist()]
            hn=add(
                "ATTEND",
                parents,
                layer=layer,
                head=head,
                target=target,
                source_positions=inds.tolist(),
                attention_values=[float(x) for x in vals.cpu()],
            )
            head_nodes.append(hn)

        merged_heads=add(
            "HEAD_MERGE",
            head_nodes,
            layer=layer,
            position=target,
        )

        mlp=add(
            "MLP",
            [resid[target]],
            layer=layer,
            position=target,
        )

        r=add(
            "RESIDUAL_ADD",
            [resid[target], merged_heads, mlp],
            layer=layer,
            position=target,
        )
        next_resid.append(r)

    resid=next_resid

final=add(
    "FINAL",
    [resid[-1]],
    predicted_token=tokenizer.decode([next_id]),
)

print("Gödel-trace nodes:", len(nodes))
print("final node:", "G"+str(final))

In [ ]:
def show(node_id, depth=0, max_depth=6):
    n=nodes[node_id]
    pad="  "*depth
    info=""
    if n.op=="TOKEN":
        info=f" {n.meta['text']!r} @pos {n.meta['position']}"
    elif n.op=="ATTEND":
        info=f" L{n.meta['layer']} H{n.meta['head']} <- positions {n.meta['source_positions']}"
    elif "layer" in n.meta:
        info=f" L{n.meta['layer']} pos {n.meta.get('position')}"
    print(f"{pad}G{node_id}: {n.op}{info}")

    if depth>=max_depth:
        return
    for p in n.parents:
        show(p, depth+1, max_depth)

show(final)

print("\n✓ זהו קוד רקורסיבי: כל תוצאה מצביעה על הפעולה ועל המקורות שיצרו אותה.")

## מה נעשה אחרי שזה רץ?

נריץ כמה משפטים שונים ונמיר כל trace לייצוג של גורמים ראשוניים/תתי-קודים.

אז נוכל לבדוק `gcd`, חליקות ומבנים משותפים — בלי לאמן שום מערכת שתזהה אותם.